# Semana 2: Tarea

**Tópicos de Finanzas Avanzadas (ECON-421, UPAO 2026-20)**
**Entrega: hasta el lunes de la semana siguiente, vía `git commit` + `push` en tu fork** (`02_costo_capital_wacc/clase02_tarea.ipynb`)

Completa las celdas marcadas con `# TU CÓDIGO AQUÍ` y las celdas de texto marcadas con .
El notebook debe correr de inicio a fin sin errores (`Kernel  Restart & Run All`).

## Parte 1: beta de tu empresa (5 pts)

Elige una empresa listada distinta de SCCO (sugerencias: BVN, IFS, o una que te interese) y estima su beta
contra el S&P 500 con 5 años de retornos mensuales. Reporta beta, error estándar y $R^2$.

In [1]:
import yfinance as yf
import statsmodels.api as sm

TICKER = "IFS"   #  tu empresa

# TU CÓDIGO AQUÍ: descarga, retornos, regresión y summary
px = yf.download([TICKER, "^GSPC"], start="2021-08-01",
    interval="1mo", auto_adjust=True, progress=False)["Close"]

px.head()

Ticker,IFS,^GSPC
Date,,
2021-08-01,17.468893,4522.680176
2021-09-01,17.252560,4307.540039
2021-10-01,22.189589,4605.379883
2021-11-01,22.405922,4567.000000
2021-12-01,20.959675,4766.180176


In [2]:
r = px.pct_change().dropna()
r = r.rename(columns={"^GSPC": "SP500"})

r.head()

Ticker,IFS,SP500
Date,,
2021-09-01,-0.012384,-0.047569
2021-10-01,0.286162,0.069144
2021-11-01,0.009749,-0.008334
2021-12-01,-0.064548,0.043613
2022-01-01,0.195224,-0.052585


In [3]:
X = sm.add_constant(r["SP500"])
modelo = sm.OLS(r[TICKER], X, missing="drop").fit()
print(modelo.summary())

beta_ols = modelo.params["SP500"]
ee = modelo.bse["SP500"]
print(f"\nBeta OLS = {beta_ols:.3f}  (error estándar {ee:.3f})")
print(f"Intervalo aproximado al 95%: [{beta_ols - 2*ee:.2f}, {beta_ols + 2*ee:.2f}]")

                            OLS Regression Results                            
Dep. Variable:                    IFS   R-squared:                       0.149
Model:                            OLS   Adj. R-squared:                  0.134
Method:                 Least Squares   F-statistic:                     10.31
Date:                Sun, 20 Sep 2026   Prob (F-statistic):            0.00214
Time:                        22:30:24   Log-Likelihood:                 64.226
No. Observations:                  61   AIC:                            -124.5
Df Residuals:                      59   BIC:                            -120.2
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0154      0.011      1.373      0.1

 **Interpreta en una oración cada uno** (estilo CFA, *holding all else constant*):

- Beta: El beta es 0.785 por lo que, manteniendo lo demás constante, si el SP 500 varía 1%, IFS tiende a variar aprox 0.785% en la misma dirección.
- Error estándar: El error estándar de 0.244 indica cierta incertidumbre en la estimación del beta, entonces 0.785 no se toma como valor exacto.
- $R^2$: El $R^2$ de 0.149 da a enteder que casi el 14.9% de la variación de retornos es explicada por movimientos del SP 500.

## Parte 2: beta ajustado y costo del equity (4 pts)

Calcula el beta ajustado de Blume y el $K_e$ por CAPM. Declara tu tasa libre de riesgo (fuente) y tu ERP (justifícala en una línea).

In [4]:
def beta_ajustado(beta_ols):
    # TU CÓDIGO AQUÍ
    pass
    """Ajuste de Blume: los betas tienden a 1 en el tiempo."""
    return 0.67 * beta_ols + 0.33

def capm(rf, beta, erp, crp=0.0, lam=1.0):
    # TU CÓDIGO AQUÍ
    pass
    """Costo del equity: CAPM con prima por riesgo país opcional."""
    return rf + beta * erp + lam * crp

# TU CÓDIGO AQUÍ: rf, ERP y Ke con tus supuestos declarados
beta_aj = beta_ajustado(beta_ols)

print(f"Beta OLS      = {beta_ols:.3f}")
print(f"Beta ajustado = {beta_aj:.3f}")

import sys; sys.path.append("..")
from utils import fred

# Tasa libre de riesgo: cierre más reciente del Treasury a 10 años
dgs10 = fred.get_series("DGS10").dropna()
rf = float(dgs10.iloc[-1]) / 100
print(f"Tasa libre de riesgo (T10Y, FRED): {rf:.2%}")

ERP = 0.055
ke = capm(rf, beta_aj, ERP)
print(f"Costo del equity (CAPM) = {ke:.2%}")

Beta OLS      = 0.785
Beta ajustado = 0.856
Tasa libre de riesgo (T10Y, FRED): 4.94%
Costo del equity (CAPM) = 9.65%


- Tasa libre de riesgo: Usé 4.94% del Treasury de EE. UU. a 10 años de FRED, porque es una referencia de largo plazo.
- ERP: Usé 5.5% porque está dentro del rango de 4% a 6% trabajado en clase para el mercado estadounidense.

## Parte 3: prima por riesgo país (3 pts)

Repite el cálculo de $K_e$ agregando una prima por riesgo país de 1.6% con $\lambda = 1$.

In [5]:
# TU CÓDIGO AQUÍ: Ke con CRP
crp = 0.016
lam = 1

ke_crp = capm(rf, beta_aj, ERP, crp=crp, lam=lam)

print(f"Ke sin riesgo país = {ke:.2%}")
print(f"Ke con riesgo país = {ke_crp:.2%}")

Ke sin riesgo país = 9.65%
Ke con riesgo país = 11.25%


 **Responde:** un proyecto de esta empresa rinde 12%. ¿Se acepta con el $K_e$ sin riesgo país? ¿Y con riesgo país?
¿Qué aprendes sobre valorar en mercados emergentes?

*(Sin riesgo país sí aceptaría el proyecto porque rinde 12% y supera el Ke de 9.65%. Con riesgo país también se acepta porque 12 es mayor que 11.25)*
*(Que se debe considerar un riesgo adicional del país ya que eso eleva el rendimiento mínimo que se le exige a la inversión.)*

## Parte 4: el WACC completo (4 pts)

Construye el WACC de tu empresa con datos de `yfinance`: capitalización bursátil, deuda total del balance,
$K_d$ aproximado por gasto de intereses y la tasa de impuestos que corresponda a su jurisdicción (decláralo).
Verifica el control de razonabilidad: $K_d(1-t) < WACC < K_e$.

In [6]:
def wacc(E, D, ke, kd, t):
    """Promedio ponderado del costo de capital, pesos a valor de mercado."""
    V = E + D
    return E / V * ke + D / V * kd * (1 - t)

tk = yf.Ticker("IFS")
E = tk.fast_info["marketCap"]
bs = tk.balance_sheet
fin = tk.financials

D = float(bs.loc["Total Debt"].iloc[0])
gasto_intereses = abs(float(fin.loc["Interest Expense"].iloc[0]))
kd = gasto_intereses / D

tax = 0.295  # tasa de IR usada para la operación peruana

# E viene en USD y los estados financieros de IFS vienen en PEN,
# por eso convierto la capitalización bursátil a soles
fx = yf.download("PEN=X", period="5d",
                 auto_adjust=True, progress=False)["Close"].dropna()

tipo_cambio = float(fx.iloc[-1].iloc[0])
E_pen = E * tipo_cambio

WACC = wacc(E_pen, D, ke, kd, tax)

print(f"Tipo de cambio USD/PEN = {tipo_cambio:.3f}")
print(f"E (market cap)         = {E_pen/1e9:,.1f} mil millones PEN")
print(f"D (deuda total)        = {D/1e9:,.1f} mil millones PEN")
print(f"Kd aproximado          = {kd:.2%}")
print(f"Tasa de impuesto       = {tax:.2%}")
print(f"WACC de IFS            = {WACC:.2%}")

print(f"\nKd(1-t) = {kd*(1-tax):.2%}")
print(f"WACC    = {WACC:.2%}")
print(f"Ke      = {ke:.2%}")

Tipo de cambio USD/PEN = 3.373
E (market cap)         = 20.6 mil millones PEN
D (deuda total)        = 11.0 mil millones PEN
Kd aproximado          = 18.78%
Tasa de impuesto       = 29.50%
WACC de IFS            = 10.90%

Kd(1-t) = 13.24%
WACC    = 10.90%
Ke      = 9.65%


- Supuestos: La capitalización bursátil se convirtió de USD a PEN para hacerla comparable con la deuda del balance. El \(Kd\) se aproximó como gasto por intereses/deuda total y se usó una tasa de impuesto de 29.5%.
- Control de razonabilidad: No se cumple el orden usual \(Kd(1-t)<WACC<Ke\), ya que el \(Kd\) aproximado de IFS resulta elevado y podría deberse a que al ser empresa financiera, el gasto por intereses forma parte importante de su operación.

## Parte 5: ítems tipo CFA (4 pts)

Responde en la celda final, justificando en una línea cada respuesta.

**1.** Para valorar en dólares los flujos de largo plazo de una empresa, la tasa libre de riesgo *más apropiada* es:
&nbsp;&nbsp;A. la tasa de política monetaria de la Fed.
&nbsp;&nbsp;B. el rendimiento del Treasury a 3 meses.
&nbsp;&nbsp;C. el rendimiento del Treasury a 10 años.

**2.** Una acción tiene $\beta = 0.8$. Si el mercado sube 10% en un mes, el modelo predice que la acción *más probablemente*:
&nbsp;&nbsp;A. subirá exactamente 8%.
&nbsp;&nbsp;B. subirá alrededor de 8% en promedio, más su componente propio.
&nbsp;&nbsp;C. subirá 10% menos la tasa libre de riesgo.

**3.** Al calcular los pesos del WACC, un analista usa los valores en libros de deuda y patrimonio. Su WACC *más probablemente* queda:
&nbsp;&nbsp;A. correcto, porque el balance está auditado.
&nbsp;&nbsp;B. distorsionado, porque los pesos deben ser a valor de mercado.
&nbsp;&nbsp;C. distorsionado solo si la empresa no paga impuestos.

**4.** Con $K_d = 8\%$ y $t = 25\%$, el costo de la deuda después de impuestos es:
&nbsp;&nbsp;A. 2.0%
&nbsp;&nbsp;B. 6.0%
&nbsp;&nbsp;C. 8.0%

 **Respuestas:**

- 1.C. Treasury a 10 años porque estamos viendo flujos de largo plazo y esa tasa es la que mejor se parece.
- 2.B. La acción tendería a subir cerca de 8% en promedio, aunque también pueden influir otros factores propios de la empresa.
- 3.B. Quedaría distorsionado porque para el WACC se deben usar valores de mercado, no los valores contables del balance.
- 4.B. Da 6%, porque al 8% se le aplica el efecto de impuestos: \(8\% \times (1-0.25)\).

---
**Recuerda:** `Kernel  Restart & Run All` antes de entregar, y luego:

```bash
git add 02_costo_capital_wacc/clase02_tarea.ipynb
git commit -m "Semana 2: tarea"
git push
```